In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import wave
import xgboost as xgb
import joblib
import threading
from PyQt5 import QtCore, QtGui, QtWidgets
from PyQt5.QtMultimedia import QSound
from PyQt5.QtSvg import QSvgWidget
import sounddevice as sd
from collections import deque

class FeatureExtractor:
    @staticmethod
    def get_mfcc(y, sr):
        if len(y) == 0:
            return np.zeros(400)
        y = librosa.resample(y, orig_sr=sr, target_sr=8000)
        y = y[:40000] if len(y) > 40000 else np.pad(y, (0, 40000 - len(y)), 'constant')
        mfcc = librosa.feature.mfcc(y=y, sr=8000, n_mfcc=10, n_fft=2048, hop_length=1024)
        return mfcc.T.flatten()

    @staticmethod
    def get_pitch(y, sr):
        if len(y) == 0:
            return 0
        pitches, voiced_flag, voiced_probs = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
        voiced_pitches = pitches[voiced_flag]  
        return np.max(voiced_pitches) if voiced_pitches.size > 0 else 0

    @staticmethod
    def extract_features(file_path):
        y, sr = librosa.load(file_path, sr=None)
        if len(y) == 0:
            raise ValueError(f"No audio data in file {file_path}")
        if np.all(y == 0):
            raise ValueError("The Microphone is not connected !")
        if np.mean(np.abs(y)) < 0.03:  # Adjust this threshold based on your needs
            raise ValueError("Recorded audio is too quiet !")
        pitch = FeatureExtractor.get_pitch(y, sr)
        mfcc_features = FeatureExtractor.get_mfcc(y, sr)
        return [pitch] + mfcc_features.tolist()

def predict_gender(model_path, imputer_path, scaler_path, encoder_path, audio_features):
    classifier = joblib.load(model_path)
    
    imputer = joblib.load(imputer_path)
    scaler = joblib.load(scaler_path)
    encoder = joblib.load(encoder_path)
    
    audio_features_df = pd.DataFrame([audio_features])
    audio_features_imputed = imputer.transform(audio_features_df)
    audio_features_scaled = scaler.transform(audio_features_imputed)
    prediction = classifier.predict(audio_features_scaled)
    predicted_gender = encoder.inverse_transform(prediction)
    
    return predicted_gender[0]

# Obtenir le répertoire de base du script
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

class CircularProgressBar(QtWidgets.QWidget):
    def __init__(self, parent=None, color=QtGui.QColor("#D10000")):  
        super().__init__(parent)
        self.value = 0
        self.duration = 5
        self.timer = QtCore.QTimer(self)
        self.timer.timeout.connect(self.update_timer)
        self.elapsed_timer = QtCore.QElapsedTimer()
        self.color = color

    def start(self):
        self.value = 0
        self.elapsed_timer.start()
        self.timer.start(16)

    def update_timer(self):
        elapsed_time = self.elapsed_timer.elapsed() / 1000.0
        self.value = elapsed_time
        self.update()
        if elapsed_time >= self.duration:
            self.timer.stop()
            self.value = self.duration  # S'assurer que la valeur finale est bien 5

    def stop(self):
        self.timer.stop()
        self.value = 0
        self.update()

    def paintEvent(self, event):
        painter = QtGui.QPainter(self)
        painter.setRenderHint(QtGui.QPainter.Antialiasing)

        rect = self.rect()
        size = min(rect.width(), rect.height())
        pen_width = size / 8
        radius = size / 2 - pen_width / 2

        square_rect = QtCore.QRectF(rect.center().x() - radius, rect.center().y() - radius, 2 * radius, 2 * radius)

        pen = QtGui.QPen(QtGui.QColor(200, 200, 200), int(pen_width))
        painter.setPen(pen)
        painter.drawEllipse(square_rect)

        pen = QtGui.QPen(self.color, int(pen_width))
        painter.setPen(pen)
        start_angle = 90 * 16
        span_angle = -int(self.value / self.duration * 360 * 16)
        painter.drawArc(square_rect, start_angle, span_angle)

        font = QtGui.QFont('Poetsen One', 12)
        painter.setFont(font)
        painter.setPen(QtGui.QColor(0, 0, 0))
        painter.drawText(square_rect, QtCore.Qt.AlignCenter, f"{int(self.value):02d} s")

class AudioLevelBar(QtWidgets.QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.levels = deque([0] * 100, maxlen=100)  # Ajuster pour 5 secondes avec une mise à jour toutes les 50ms
        self.setMinimumHeight(120)  # Augmenter la hauteur pour la longueur des barres
        self.timer = QtCore.QTimer(self)
        self.timer.timeout.connect(self.update_levels)
        self.playback_timer = QtCore.QTimer(self)
        self.playback_timer.timeout.connect(self.update_playback)
        self.playback_position = len(self.levels)
        self.recording_thread = None  # Reference to the recording thread

    def set_recording_thread(self, recording_thread):
        self.recording_thread = recording_thread

    def update_levels(self):
        if self.recording_thread and self.recording_thread.isRunning():
            if self.recording_thread.audio_levels:
                level = self.recording_thread.audio_levels[-1]  # Get the last level
                self.levels.append(level)
        else:
            self.timer.stop()
        self.update()

    def start_playback(self):
        self.playback_position = len(self.levels)
        self.playback_timer.start(50)

    def update_playback(self):
        if self.playback_position > 0:
            self.playback_position -= 1
            self.update()
        else:
            self.playback_timer.stop()
            self.playback_position = len(self.levels)
            self.update()

    def paintEvent(self, event):
        painter = QtGui.QPainter(self)
        painter.setRenderHint(QtGui.QPainter.Antialiasing)
        bar_width = self.width() / len(self.levels)
        mid_y = self.height() // 2

        # Remplissage gris très clair
        painter.fillRect(self.rect(), QtGui.QColor("#f0f0f0"))

        for i, level in enumerate(self.levels):
            rect = QtCore.QRectF(i * bar_width, mid_y, bar_width - 3, -self.height() * level / 2)
            rect_neg = QtCore.QRectF(i * bar_width, mid_y, bar_width - 3, self.height() * level / 2)
            painter.fillRect(rect, QtGui.QColor("#444444"))  # Couleur sobre pour les barres
            painter.fillRect(rect_neg, QtGui.QColor("#444444"))

        # Dessiner la barre verticale rouge pour la lecture
        if self.playback_position < len(self.levels):
            red_bar_x = self.playback_position * bar_width
            painter.fillRect(int(red_bar_x), 0, 6, self.height(), QtGui.QColor("red"))  # Augmenter la largeur de la barre rouge

# Utilisation du QElapsedTimer dans la gestion de l'enregistrement pour s'assurer que l'enregistrement ne dépasse pas les 5 secondes
class RecordThread(QtCore.QThread):
    recorded = QtCore.pyqtSignal(str)
    error = QtCore.pyqtSignal(str)
    update_ui = QtCore.pyqtSignal(float, float)

    def __init__(self, output_path, duration=5, file_name='test.wav'):
        super().__init__()
        self.output_path = output_path
        self.duration = duration
        self.file_name = file_name
        self.stop_recording_event = threading.Event()
        self.elapsed_timer = QtCore.QElapsedTimer()
        self.frames = []
        self.audio_levels = deque(maxlen=100)  # Shared audio levels

    def callback(self, indata, frames, time, status):
        if status:
            print(status, file=sys.stderr)
        if self.stop_recording_event.is_set():
            raise sd.CallbackStop
        self.frames.append(indata.copy())

        audio_data = np.abs(indata).mean()
        level = audio_data / 32768
        self.audio_levels.append(level)

        elapsed_time = self.elapsed_timer.elapsed() / 1000.0
        self.update_ui.emit(level, elapsed_time)

    def run(self):
        self.elapsed_timer.start()
        self.frames = []

        try:
            with sd.InputStream(samplerate=22050, channels=1, callback=self.callback, dtype='int16'):
                while self.elapsed_timer.elapsed() / 1000.0 < self.duration:
                    QtCore.QThread.msleep(50)
        except Exception as e:
            self.error.emit(str(e))
            return

        if not self.frames:
            self.error.emit("No audio recorded.")
            return

        os.makedirs(self.output_path, exist_ok=True)
        file_path = os.path.join(self.output_path, self.file_name)
        with wave.open(file_path, 'wb') as wave_file:
            wave_file.setnchannels(1)
            wave_file.setsampwidth(2)
            wave_file.setframerate(22050)
            wave_file.writeframes(b''.join([frame.tobytes() for frame in self.frames]))

        self.recorded.emit(file_path)

    def stop(self):
        self.stop_recording_event.set()

class PredictionThread(QtCore.QThread):
    prediction_made = QtCore.pyqtSignal(str)
    error_occurred = QtCore.pyqtSignal(str)

    def __init__(self, model_path, imputer_path, scaler_path, encoder_path, audio_features):
        super().__init__()
        self.model_path = model_path
        self.imputer_path = imputer_path
        self.scaler_path = scaler_path
        self.encoder_path = encoder_path
        self.audio_features = audio_features

    def run(self):
        try:
            classifier = joblib.load(self.model_path)
            imputer = joblib.load(self.imputer_path)
            scaler = joblib.load(self.scaler_path)
            encoder = joblib.load(self.encoder_path)

            audio_features_df = pd.DataFrame([self.audio_features])
            audio_features_imputed = imputer.transform(audio_features_df)
            audio_features_scaled = scaler.transform(audio_features_imputed)
            prediction = classifier.predict(audio_features_scaled)
            predicted_gender = encoder.inverse_transform(prediction)

            self.prediction_made.emit(predicted_gender[0])
        except Exception as e:
            self.error_occurred.emit(str(e))

class GenderClassificationApp(QtWidgets.QWidget):
    def __init__(self):
        super().__init__()
        self.initUI()
        self.audio_file_path = None
        self.is_recording = False
        self.recording_thread = None
        self.playback_timer = QtCore.QTimer(self)  # Add the playback timer
        self.playback_timer.timeout.connect(self.clear_playing_text)  # Connect the timer to the slot

    def initUI(self):
        QtGui.QFontDatabase.addApplicationFont(os.path.join(BASE_DIR, 'fonts', 'PoetsenOne-Regular.ttf'))
        custom_font = 'Poetsen One'
        
        self.setWindowTitle("Real-Time Gender Classification")
        self.setGeometry(100, 100, 400, 400)
        self.setStyleSheet("background-color: #f0f0f0;")
        
        layout = QtWidgets.QVBoxLayout()
        
        self.title_label = QtWidgets.QLabel("Gender Identification")
        self.title_label.setFont(QtGui.QFont(custom_font, 22, QtGui.QFont.Bold))
        self.title_label.setAlignment(QtCore.Qt.AlignCenter)
        layout.addWidget(self.title_label)
        
        self.progress_bar = AudioLevelBar()
        layout.addWidget(self.progress_bar)
        
        self.result_label = QtWidgets.QLabel("Plug in your microphone and click on RECORD")
        self.result_label.setFont(QtGui.QFont(custom_font, 12))
        self.result_label.setAlignment(QtCore.Qt.AlignCenter)
        self.result_label.setStyleSheet("color: #02020B;")
        layout.addWidget(self.result_label)

        self.circular_progress_bar = CircularProgressBar()
        layout.addWidget(self.circular_progress_bar)
        
        buttons_layout = QtWidgets.QHBoxLayout()
        
        play_layout = QtWidgets.QVBoxLayout()
        self.play_button = QSvgWidget(os.path.join(BASE_DIR, 'icons', 'play_arrow_48dp_FILL0_wght400_GRAD0_opsz48.svg'))
        self.play_button.setFixedSize(70, 70)
        self.play_button.setStyleSheet("background-color: transparent;")
        self.play_button.mousePressEvent = self.play_recording
        play_layout.addWidget(self.play_button, alignment=QtCore.Qt.AlignCenter)
        
        play_label = QtWidgets.QLabel("PLAY")
        play_label.setFont(QtGui.QFont(custom_font, 10))  # Smaller font size
        play_label.setAlignment(QtCore.Qt.AlignCenter)
        play_label.setStyleSheet("color: #000000;")  # Black color
        play_layout.addWidget(play_label, alignment=QtCore.Qt.AlignCenter)

        buttons_layout.addLayout(play_layout)
        
        record_layout = QtWidgets.QVBoxLayout()
        self.record_button = QSvgWidget(os.path.join(BASE_DIR, 'icons', 'settings_voice_48dp_FILL0_wght400_GRAD0_opsz48(1).svg'))
        self.record_button.setFixedSize(80, 80)
        self.record_button.setStyleSheet("background-color: transparent;")
        self.record_button.mousePressEvent = self.start_recording
        record_layout.addWidget(self.record_button, alignment=QtCore.Qt.AlignCenter)
        
        record_label = QtWidgets.QLabel("RECORD")
        record_label.setFont(QtGui.QFont(custom_font, 10))  # Smaller font size
        record_label.setAlignment(QtCore.Qt.AlignCenter)
        record_label.setStyleSheet("color: #000000;")  # Black color
        record_layout.addWidget(record_label, alignment=QtCore.Qt.AlignCenter)
        
        buttons_layout.addLayout(record_layout)

        stop_layout = QtWidgets.QVBoxLayout()
        self.stop_button = QSvgWidget(os.path.join(BASE_DIR, 'icons', 'stop_48dp_FILL0_wght400_GRAD0_opsz48.svg'))
        self.stop_button.setFixedSize(60, 60)
        self.stop_button.setStyleSheet("background-color: transparent;")
        self.stop_button.mousePressEvent = self.stop_recording
        stop_layout.addWidget(self.stop_button, alignment=QtCore.Qt.AlignCenter)
        
        stop_label = QtWidgets.QLabel("STOP")
        stop_label.setFont(QtGui.QFont(custom_font, 10))  # Smaller font size
        stop_label.setAlignment(QtCore.Qt.AlignCenter)
        stop_label.setStyleSheet("color: #000000;")  # Black color
        stop_layout.addWidget(stop_label, alignment=QtCore.Qt.AlignCenter)

        buttons_layout.addLayout(stop_layout)
        
        layout.addLayout(buttons_layout)
        
        self.setLayout(layout)

    def set_error_message(self, message):
        self.result_label.setText(message)
        self.result_label.setStyleSheet("color: #D10000;")  # Noir
        
    def start_recording(self, event):
        if not self.is_recording:
            self.is_recording = True
            self.result_label.setText("Recording... Please speak clearly for best results")
            self.result_label.setStyleSheet("color: #02020B;")
            QtWidgets.QApplication.processEvents()

            self.recording_thread = RecordThread(os.path.join(BASE_DIR, 'recorded_audio'))
            self.progress_bar.set_recording_thread(self.recording_thread)
            self.recording_thread.update_ui.connect(self.update_ui)  # Connect the new signal
            self.recording_thread.recorded.connect(self.on_recording_finished)
            self.recording_thread.error.connect(self.on_recording_error)
            self.recording_thread.start()
            self.circular_progress_bar.start()
            self.progress_bar.timer.start(50)  # Start the timer for updating audio levels
        else:
            self.stop_recording()

    @QtCore.pyqtSlot(float, float)
    def update_ui(self, level, elapsed_time):
        self.circular_progress_bar.value = elapsed_time
        self.circular_progress_bar.update()

    def on_recording_finished(self, file_path):
        if self.is_recording:
            self.audio_file_path = file_path
            self.is_recording = False
            self.progress_bar.timer.stop()  # Stop the timer when recording is finished
            try:
                audio_features = FeatureExtractor.extract_features(self.audio_file_path)
                model_path = os.path.join(BASE_DIR, 'xgboost_model.pkl')  # Utiliser xgboost_model.pkl
                imputer_path = os.path.join(BASE_DIR, 'imputer.joblib')
                scaler_path = os.path.join(BASE_DIR, 'scaler.joblib')
                encoder_path = os.path.join(BASE_DIR, 'encoder.joblib')
                
                self.prediction_thread = PredictionThread(model_path, imputer_path, scaler_path, encoder_path, audio_features)
                self.prediction_thread.prediction_made.connect(self.show_prediction)
                self.prediction_thread.error_occurred.connect(self.set_error_message)
                self.prediction_thread.start()
            except ValueError as e:
                self.set_error_message(str(e))

    def show_prediction(self, predicted_gender):
        gender_text = f"Predicted Gender: {predicted_gender.upper()}"
        if predicted_gender == 'female':
            self.result_label.setTextFormat(QtCore.Qt.RichText)
            self.result_label.setText(f"<span style='font-size:22px;'>Predicted Gender: <span style='color:#A30000; font-size:22px;'>FEMALE <img src='{os.path.join(BASE_DIR, 'icons', 'female_48dp_FILL0_wght400_GRAD0_opsz48.svg')}' width='20' height='20'></span></span>")
        elif predicted_gender == 'male':
            self.result_label.setTextFormat(QtCore.Qt.RichText)
            self.result_label.setText(f"<span style='font-size:22px;'>Predicted Gender: <span style='color:#002F77; font-size:22px;'>MALE <img src='{os.path.join(BASE_DIR, 'icons', 'male_48dp_FILL0_wght400_GRAD0_opsz48.svg')}' width='20' height='20'></span></span>")
        else:
            self.result_label.setText(gender_text)
            self.result_label.setStyleSheet("color: #000000;")

        self.result_label.setStyleSheet("color: #02020B;")
        self.result_label.setAlignment(QtCore.Qt.AlignCenter)
        
    def on_recording_error(self, error_message):
        self.is_recording = False
        self.progress_bar.timer.stop()  # Stop the timer in case of error
        self.set_error_message(error_message)

    def stop_recording(self, event=None):
        if self.is_recording:
            self.recording_thread.stop()
            self.is_recording = False
            self.recording_thread.wait()
            self.circular_progress_bar.stop()
            self.progress_bar.timer.stop()
            self.result_label.setText('Recording has been stopped.')
            self.result_label.setStyleSheet("color: black;")

    def play_recording(self, event):
        if self.audio_file_path:
            QSound.play(self.audio_file_path)
            self.result_label.setText("Playing recording...")
            self.playback_timer.start(5000)  # Start the timer with a duration of 5 seconds
            self.progress_bar.start_playback()
        else:
            QtWidgets.QMessageBox.warning(self, "Warning", "No recording available to play.")
            self.result_label.setText("No recording available to play.")
            self.result_label.setStyleSheet("color: black;")
            
    def clear_playing_text(self):
        self.result_label.setText("Record again...")  # Set the text to "Record again" after playback
        self.result_label.setStyleSheet("color: #000000;")  # Set text color to black
        self.playback_timer.stop()  # Stop the timer

def main():
    app = QtWidgets.QApplication([])
    window = GenderClassificationApp()
    window.show()
    app.exec_()

if __name__ == '__main__':
    main()